# 01 ? Your first ecommerce agent

Learn the relationship between an Agent, a Runner, a Session and events. Instructions describe behavior; the Runner manages each invocation.

**Runtime:** ADK 2.10.0, your existing `dataengenv` kernel and Application Default Credentials (ADC).

**Cost:** One small Gemini call. Run cells from top to bottom. This notebook is self-contained; it does not need `course.py` or `config.json`.

## Initialize

Set your Google Cloud project, location and model below. The client uses standard TLS certificate verification.


In [ ]:
from google.adk.agents import Agent
from google.adk.models import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

PROJECT = "gen-lang-client-0367558705"
LOCATION = "global"
MODEL = "gemini-2.5-flash-lite"
APP_NAME = "ecommerce_lab"
USER_ID = "customer_101"

model = Gemini(
    model=MODEL,
    client_kwargs={"vertexai": True, "project": PROJECT, "location": LOCATION},
)


In [ ]:
agent = Agent(
    name="shop_assistant",
    model=model,
    instruction="You explain ecommerce concepts in two short sentences. Do not invent order records.",
    generate_content_config=types.GenerateContentConfig(
        temperature=0, max_output_tokens=768,
    ),
)

session_service = InMemorySessionService()
session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID)
runner = Runner(agent=agent, app_name=APP_NAME, session_service=session_service)

message = types.Content(
    role="user",
    parts=[types.Part(text="Explain the difference between an order and an invoice.")],
)
events = []
async for event in runner.run_async(
    user_id=USER_ID, session_id=session.id, new_message=message,
):
    events.append(event)
    if event.error_code:
        raise RuntimeError(f"{event.error_code}: {event.error_message}")


The **Agent** defines the model and instructions. The **Session** holds the conversation in memory. The **Runner** sends the message through the agent and yields **events**, including the final answer.


In [ ]:
print("Session:", session.id)
print("Event authors:", [event.author for event in events])

for event in events:
    if event.is_final_response() and event.content:
        answer = "".join(part.text or "" for part in event.content.parts or [])
        print("Final answer:", answer)

print("Tokens reported:", sum(
    event.usage_metadata.total_token_count or 0
    for event in events
    if event.usage_metadata and not event.partial
))


## Try it

Change the instruction to explain things to a warehouse operator, then recreate the agent and session.


## Reference

[Official ADK documentation](https://adk.dev/). Configuration is in the first code cell. Use your existing authenticated kernel; no environment activation or login cells are needed.
